# Data Cleaning Pipeline

This notebook handles the **full extraction and cleaning** of champion position data from minimap screenshots.

**Pipeline:**
1. Load the YOLO minimap model
2. Auto-discover all match folders in `Dataset/`
3. For each match, for each team (Blue + All):
   - Run YOLO inference on all screenshots → raw `match_data.csv`
   - Clean: deduplicate, purge hallucinated champions, fill gaps, assign roles
   - Save `cleaned_match_data.csv`
4. Matches that already have cleaned data for both teams are **skipped** (incremental)

Screenshots are located at: `C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset`

## 1. Setup & Model Loading

In [8]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import re
import torch
from ultralytics import YOLO
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────────────
DATASET_ROOT = Path(r"C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset")

# ── YOLO Model ───────────────────────────────────────────────────────
from huggingface_hub import hf_hub_download

repo_id   = "Ultralytics/YOLO11"     # Only matters for first-time download
weight_file = "yolo11m-minimap.pt"

model_dir = Path.cwd() / "models"
model_dir.mkdir(parents=True, exist_ok=True)
local_model_path = model_dir / weight_file

if local_model_path.exists():
    print(f"Using local model: {local_model_path}")
    model_path = local_model_path
else:
    print(f"Downloading {weight_file} to {model_dir}...")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=weight_file,
        local_dir=model_dir
    )
    model_path = Path(downloaded_path)
    print(f"Saved model to {model_path}")

print(f"Loading weights from {model_path}...")
model = YOLO(str(model_path))

# ── CUDA Check ───────────────────────────────────────────────────────
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Using local model: c:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\pyLoL\models\yolo11m-minimap.pt
Loading weights from c:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\pyLoL\models\yolo11m-minimap.pt...
CUDA Available: True
Device Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 2. Configuration

In [9]:
# ── Inference Parameters ──────────────────────────────────────────
CHUNK_SIZE  = 128   # Feeds the GPU this many images at a time
BATCH_SIZE  = 8     # Tensor-Core-friendly batch size
CONFIDENCE  = 0.2   # Detection confidence threshold
TEAMS       = ["Blue", "All"]  # Process both teams per match

# ── Cleaning Parameters ───────────────────────────────────────────
EARLY_GAME_LIMIT = 600       # ~10 minutes of frames for role assignment
FOUNTAIN_RADIUS  = 70        # Pixel radius around fountain for team detection
MAP_SIZE         = 512       # Minimap is 512x512
LOW_CONF_CUTOFF  = 0.6       # Confidence below this is considered a hallucination

# ── Role Rosters (for bot/support disambiguation) ─────────────────
ADC_ROSTER = {
    'Aphelios', 'Ashe', 'Caitlyn', 'Draven', 'Ezreal', 'Jhin', 'Jinx', 
    'Kaisa', 'Kalista', 'KogMaw', 'Lucian', 'MissFortune', 'Nilah', 
    'Samira', 'Sivir', 'Smolder', 'Tristana', 'Twitch', 'Varus', 'Vayne', 'Xayah', 'Zeri',
    'Lux', 'Yasuo', 'Ziggs'
}

SUPPORT_ROSTER = {
    'Alistar', 'Bard', 'Blitzcrank', 'Brand', 'Braum', 'Janna', 'Karma', 
    'Leona', 'Lulu', 'Milio', 'Morgana', 'Nami', 'Nautilus', 'Pyke', 
    'Rakan', 'Rell', 'Renata', 'Senna', 'Seraphine', 'Sona', 'Soraka', 
    'TahmKench', 'Taric', 'Thresh', 'Velkoz', 'Xerath', 'Yuumi', 'Zilean', 'Zyra', 'Camille'
}

# ── Ground-Truth Role Assignment (Riot API from .env) ──────────────
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
API_KEY = os.getenv("RIOT_API_KEY") or os.getenv("API_KEY", "")
ROUTING_REGION = "sea"

if not API_KEY:
    print("[!] Warning: RIOT_API_KEY not found in .env! (Refresh at https://developer.riotgames.com/)")
else:
    print(f"[OK] Loaded Riot API Key from .env: {API_KEY[:10]}...")

[OK] Loaded Riot API Key from .env: RGAPI-ccbe...


## 3. Auto-Discover Matches & Check Existing Data

In [10]:
# Discover all match folders (SG2-* or SG2_*)
all_match_dirs = sorted([
    d for d in DATASET_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("SG2")
])

print(f"Found {len(all_match_dirs)} match folders in Dataset/\n")

# Determine which matches need processing
matches_to_process = []
matches_skipped    = []

for match_dir in all_match_dirs:
    match_id = match_dir.name
    all_clean = True
    for team in TEAMS:
        cleaned_csv = match_dir / f"{team} data" / "cleaned_match_data.csv"
        if not cleaned_csv.exists():
            all_clean = False
            break
    
    if all_clean:
        matches_skipped.append(match_id)
    else:
        matches_to_process.append(match_id)

print(f"  Skipping (already cleaned): {len(matches_skipped)}")
print(f"  To process:                 {len(matches_to_process)}")

if matches_to_process:
    print(f"\nMatches to process:")
    for m in matches_to_process:
        print(f"  - {m}")

Found 47 match folders in Dataset/

  Skipping (already cleaned): 47
  To process:                 0


## 4. Helper Functions

In [11]:
def extract_frame_number(filepath):
    """Extract the numeric frame number from a screenshot filename like '123.png'."""
    match = re.search(r'(\d+)\.png', str(filepath))
    return int(match.group(1)) if match else -1


def run_yolo_inference(match_id, team):
    """
    Run YOLO detection on all screenshots for a match/team.
    Returns a DataFrame with columns: [frame, champion_id, champion_name, x, y, confidence]
    Also saves the raw CSV to Dataset/{match_id}/{team} data/match_data.csv
    """
    folder_path = DATASET_ROOT / match_id / "0" / team
    
    if not folder_path.exists():
        print(f"    ⚠ Screenshot folder not found: {folder_path}")
        return None
    
    image_paths = sorted(folder_path.glob("*.png"), key=extract_frame_number)
    image_paths_str = [str(p) for p in image_paths]
    
    if not image_paths_str:
        print(f"    ⚠ No screenshots found in {folder_path}")
        return None
    
    detection_data = []
    
    with tqdm(total=len(image_paths_str), desc=f"    YOLO ({team})", leave=False) as pbar:
        for i in range(0, len(image_paths_str), CHUNK_SIZE):
            chunk = image_paths_str[i:i + CHUNK_SIZE]
            
            # batch=8 uses Tensor Cores efficiently. Don't set imgsz= (already 512x512)
            results = model(chunk, stream=True, batch=BATCH_SIZE, conf=CONFIDENCE, device=0, verbose=False)
            
            for r in results:
                frame_number = extract_frame_number(r.path)
                
                for box in r.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    class_id = int(box.cls)
                    conf = float(box.conf)
                    champion_name = model.names[class_id]
                    
                    center_x = (x1 + x2) / 2
                    center_y = (y1 + y2) / 2
                    
                    detection_data.append({
                        "frame": frame_number,
                        "champion_id": class_id,
                        "champion_name": champion_name,
                        "x": round(center_x, 1),
                        "y": round(center_y, 1),
                        "confidence": round(conf, 3)
                    })
                
                pbar.update(1)
    
    df = pd.DataFrame(detection_data)
    
    # Save raw CSV
    csv_output = DATASET_ROOT / match_id / f"{team} data" / "match_data.csv"
    csv_output.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_output, index=False)
    
    return df


def get_match_roles(match_id, api_key=None, routing_region="sea"):
    """
    Retrieves official ground-truth roles for a match:
    1. Checks for local cached 'roles.json' in Dataset/{match_id}/
    2. Checks 'player_states.json' in Dataset/{match_id}/
    3. Queries Riot Match-V5 API (if api_key provided) and caches to 'roles.json'
    Returns dict {champion_name: 'Blue_Top', ...} or None if unavailable.
    """
    match_folder = DATASET_ROOT / match_id
    roles_cache_file = match_folder / "roles.json"
    pos_map = {"TOP": "Top", "JUNGLE": "Jungle", "MIDDLE": "Mid", "BOTTOM": "Bot", "UTILITY": "Support"}

    # 1. Check local cached roles.json
    if roles_cache_file.exists():
        try:
            with open(roles_cache_file, "r", encoding="utf-8") as f:
                roles = json.load(f)
                if len(roles) >= 10:
                    return roles
        except Exception:
            pass

    # 2. Check local player_states.json (from scraper)
    states_file = match_folder / "player_states.json"
    if states_file.exists():
        try:
            with open(states_file, "r", encoding="utf-8") as f:
                states_data = json.load(f)
                if states_data:
                    first_frame = next(iter(states_data.values()))
                    roles = {}
                    for champ, info in first_frame.items():
                        team_prefix = "Blue" if info.get("team") == "ORDER" else ("Red" if info.get("team") == "CHAOS" else "")
                        pos_suffix = pos_map.get(str(info.get("position", "")).upper(), "")
                        if team_prefix and pos_suffix:
                            roles[champ] = f"{team_prefix}_{pos_suffix}"
                    if len(roles) >= 10:
                        with open(roles_cache_file, "w", encoding="utf-8") as cf:
                            json.dump(roles, cf, indent=2)
                        return roles
        except Exception:
            pass

    # 3. Query Riot Match-V5 API
    if api_key:
        clean_id = str(match_id).replace("-", "_")
        url = f"https://{routing_region}.api.riotgames.com/lol/match/v5/matches/{clean_id}?api_key={api_key}"
        try:
            import requests
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                participants = data.get("info", {}).get("participants", [])
                roles = {}
                for p in participants:
                    champ = p.get("championName", "")
                    team_prefix = "Blue" if p.get("teamId") == 100 else "Red"
                    pos_suffix = pos_map.get(str(p.get("teamPosition", "")).upper(), "Mid")
                    if champ:
                        roles[champ] = f"{team_prefix}_{pos_suffix}"
                if len(roles) >= 10:
                    match_folder.mkdir(parents=True, exist_ok=True)
                    with open(roles_cache_file, "w", encoding="utf-8") as cf:
                        json.dump(roles, cf, indent=2)
                    print(f"    ✓ Fetched and cached ground-truth roles for {match_id} via Riot API")
                    return roles
            elif res.status_code == 401:
                print("    ⚠️ Riot API Key is expired (401 Unauthorized).")
        except Exception as e:
            print(f"    ⚠️ Notice fetching roles from Riot API: {e}")

    return None


def clean_detections(df, match_id=None, api_key=None, routing_region="sea"):
    """
    Full cleaning pipeline:
    1. Deduplicate (keep highest confidence per frame/champion)
    2. Purge hallucinations (low confidence + base-structure ghosts)
    3. Keep only the true 10 champions
    4. Fill missing frames with sentinel values (-1, -1, 0)
    5. Assign roles (prioritizes ground-truth from Riot API / cached roles.json)
    
    Returns the cleaned DataFrame, or None if data is insufficient.
    """
    if df is None or df.empty:
        return None
    
    # ── STEP 1: Deduplicate ──────────────────────────────────────────
    df = df.sort_values(by=['frame', 'champion_id', 'confidence'], ascending=[True, True, False])
    df = df.drop_duplicates(subset=['frame', 'champion_id'], keep='first')
    
    # ── STEP 2: Purge hallucinations ─────────────────────────────────
    bad_detections = (
        (df['champion_name'] == 'Tristana') & 
        (df['x'] < 70) & 
        (df['y'] > 440) | 
        (df['confidence'] < LOW_CONF_CUTOFF)
    )
    df = df[~bad_detections]
    
    # ── STEP 3: Keep only the true 10 champions ──────────────────────
    champ_counts = df['champion_name'].value_counts()
    
    if len(champ_counts) < 10:
        print(f"    ⚠ Only found {len(champ_counts)} champions (need 10). Skipping.")
        return None
    
    true_10_champs = champ_counts.nlargest(10).index.tolist()
    df = df[df['champion_name'].isin(true_10_champs)]
    
    # ── STEP 4: Fill missing frames ──────────────────────────────────
    true_champs = df['champion_id'].unique()
    all_frames = range(df['frame'].min(), df['frame'].max() + 1)
    
    multi_idx = pd.MultiIndex.from_product(
        [all_frames, true_champs], 
        names=['frame', 'champion_id']
    )
    
    df_full = df.set_index(['frame', 'champion_id']).reindex(multi_idx).reset_index()
    
    df_full['x'] = df_full['x'].fillna(-1)
    df_full['y'] = df_full['y'].fillna(-1)
    df_full['confidence'] = df_full['confidence'].fillna(0)
    
    # Restore champion names
    champ_map = df.drop_duplicates('champion_id').set_index('champion_id')['champion_name'].to_dict()
    df_full['champion_name'] = df_full['champion_id'].map(champ_map)
    
    df_full = df_full.sort_values(by=['frame', 'champion_id']).reset_index(drop=True)
    
    # ── STEP 5: Assign roles ─────────────────────────────────────────
    df_full = assign_roles(df_full, match_id=match_id, api_key=api_key, routing_region=routing_region)
    
    return df_full


def assign_roles(df_full, match_id=None, api_key=None, routing_region="sea"):
    """
    Assign team (Blue/Red) and lane role to each champion.
    Prioritizes official ground-truth roles (from cached roles.json or Riot API),
    and falls back to spatial heuristics only if ground truth is unavailable.
    """
    # 1. Attempt Ground-Truth Assignment
    if match_id:
        truth_roles = get_match_roles(match_id, api_key=api_key, routing_region=routing_region)
        if truth_roles and len(truth_roles) >= 10:
            df_full['role'] = df_full['champion_name'].map(truth_roles)
            if not df_full['role'].isna().any():
                ordered_roles = [
                    'Blue_Top', 'Blue_Jungle', 'Blue_Mid', 'Blue_Bot', 'Blue_Support',
                    'Red_Top', 'Red_Jungle', 'Red_Mid', 'Red_Bot', 'Red_Support'
                ]
                df_full['role'] = pd.Categorical(df_full['role'], categories=ordered_roles, ordered=True)
                df_full = df_full.sort_values(['frame', 'role']).reset_index(drop=True)
                return df_full
            else:
                print("    ⚠️ Ground-truth mapping had unmapped champions. Falling back to heuristic.")

    # 2. Fallback to Spatial Heuristics
    return assign_roles_heuristic(df_full)


def assign_roles_heuristic(df_full):

    """
    Assign team (Blue/Red) and lane role to each champion using:
    - Fountain proximity for team assignment
    - Spatial median + variance for lane assignment
    - ADC/Support roster for bot lane disambiguation
    """
    # Filter valid early-game positions
    valid_positions = df_full[
        (df_full['x'] != -1) & 
        (df_full['frame'] <= EARLY_GAME_LIMIT)
    ]
    
    if valid_positions.empty:
        print("    ⚠ No valid early-game positions found. Skipping role assignment.")
        return None
    
    # Calculate spatial metrics per champion
    roles_df = valid_positions.groupby(['champion_id', 'champion_name']).agg(
        median_x=('x', 'median'),
        median_y=('y', 'median'),
        var_x=('x', 'var'),
        var_y=('y', 'var')
    ).reset_index()
    
    roles_df['total_variance'] = roles_df['var_x'] + roles_df['var_y']
    roles_df['side_metric'] = roles_df['median_x'] - roles_df['median_y']   # Neg=Blue, Pos=Red
    roles_df['lane_metric'] = roles_df['median_x'] + roles_df['median_y']   # Low=Top, High=Bot
    
    # ── Team assignment via fountain proximity ────────────────────────
    blue_team_ids = set()
    red_team_ids  = set()
    
    for champ_id in roles_df['champion_id'].values:
        champ_data = valid_positions[valid_positions['champion_id'] == champ_id]
        
        near_blue = ((champ_data['x'] < FOUNTAIN_RADIUS) & 
                     (champ_data['y'] > (MAP_SIZE - FOUNTAIN_RADIUS))).any()
        
        near_red  = ((champ_data['x'] > (MAP_SIZE - FOUNTAIN_RADIUS)) & 
                     (champ_data['y'] < FOUNTAIN_RADIUS)).any()
        
        if near_blue:
            blue_team_ids.add(champ_id)
        elif near_red:
            red_team_ids.add(champ_id)
    
    # Fill in remaining unassigned champions
    all_ids = set(roles_df['champion_id'].values)
    if len(blue_team_ids) == 5:
        red_team_ids = all_ids - blue_team_ids
    elif len(red_team_ids) == 5:
        blue_team_ids = all_ids - red_team_ids
    else:
        unassigned = all_ids - blue_team_ids - red_team_ids
        for cid in unassigned:
            if len(blue_team_ids) < 5:
                blue_team_ids.add(cid)
            else:
                red_team_ids.add(cid)
    
    roles_df['team'] = roles_df['champion_id'].apply(
        lambda x: 'Blue' if x in blue_team_ids else 'Red'
    )
    
    # ── Lane assignment ───────────────────────────────────────────────
    def assign_team_roles(team_df):
        team_df = team_df.sort_values('lane_metric').reset_index(drop=True)
        
        # Middle two players → distinguish mid vs jungle by variance
        mid_jg = team_df.iloc[1:3].sort_values('total_variance')
        mid_idx = mid_jg.index[0]
        jg_idx  = mid_jg.index[1]
        
        team_name = team_df['team'].iloc[0]
        roles = {}
        
        roles[team_df.loc[0, 'champion_id']] = f"{team_name}_Top"
        roles[team_df.loc[mid_idx, 'champion_id']] = f"{team_name}_Mid"
        roles[team_df.loc[jg_idx, 'champion_id']]  = f"{team_name}_Jungle"
        
        # Bot / Support disambiguation
        bot_idx_1, bot_idx_2 = 3, 4
        champ_1_name = team_df.loc[bot_idx_1, 'champion_name']
        champ_1_id   = team_df.loc[bot_idx_1, 'champion_id']
        champ_2_name = team_df.loc[bot_idx_2, 'champion_name']
        champ_2_id   = team_df.loc[bot_idx_2, 'champion_id']
        
        if champ_1_name in ADC_ROSTER and champ_2_name not in ADC_ROSTER:
            roles[champ_1_id] = f"{team_name}_Bot"
            roles[champ_2_id] = f"{team_name}_Support"
        elif champ_2_name in ADC_ROSTER and champ_1_name not in ADC_ROSTER:
            roles[champ_2_id] = f"{team_name}_Bot"
            roles[champ_1_id] = f"{team_name}_Support"
        elif champ_1_name in SUPPORT_ROSTER and champ_2_name not in SUPPORT_ROSTER:
            roles[champ_1_id] = f"{team_name}_Support"
            roles[champ_2_id] = f"{team_name}_Bot"
        elif champ_2_name in SUPPORT_ROSTER and champ_1_name not in SUPPORT_ROSTER:
            roles[champ_2_id] = f"{team_name}_Support"
            roles[champ_1_id] = f"{team_name}_Bot"
        else:
            roles[champ_1_id] = f"{team_name}_Bot"
            roles[champ_2_id] = f"{team_name}_Support"
        
        team_df['role'] = team_df['champion_id'].map(roles)
        return team_df
    
    blue_team = assign_team_roles(roles_df[roles_df['team'] == 'Blue'].copy())
    red_team  = assign_team_roles(roles_df[roles_df['team'] == 'Red'].copy())
    
    final_roles = pd.concat([blue_team, red_team])
    
    # Map roles back to the full DataFrame
    role_mapping = final_roles.set_index('champion_id')['role'].to_dict()
    df_full['role'] = df_full['champion_id'].map(role_mapping)
    
    # Force consistent ordering every frame
    ordered_roles = [
        'Blue_Top', 'Blue_Jungle', 'Blue_Mid', 'Blue_Bot', 'Blue_Support',
        'Red_Top', 'Red_Jungle', 'Red_Mid', 'Red_Bot', 'Red_Support'
    ]
    df_full['role'] = pd.Categorical(df_full['role'], categories=ordered_roles, ordered=True)
    df_full = df_full.sort_values(['frame', 'role']).reset_index(drop=True)
    
    return df_full

## 5. Run Full Pipeline (All Matches)

In [7]:
import traceback

# ── Process all matches ──────────────────────────────────────────────
results_summary = []

for idx, match_id in enumerate(matches_to_process):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(matches_to_process)}] Processing: {match_id}")
    print(f"{'='*60}")
    
    match_result = {"match_id": match_id, "status": "success", "teams": {}}
    
    try:
        for team in TEAMS:
            cleaned_csv = DATASET_ROOT / match_id / f"{team} data" / "cleaned_match_data.csv"
            
            # Skip this specific team if already cleaned
            if cleaned_csv.exists():
                print(f"  [{team}] Already cleaned. Skipping.")
                match_result["teams"][team] = "skipped"
                continue
            
            print(f"  [{team}] Running YOLO inference...")
            
            # Step A: YOLO Inference
            df_raw = run_yolo_inference(match_id, team)
            
            if df_raw is None or df_raw.empty:
                print(f"  [{team}] ✗ No detections. Skipping.")
                match_result["teams"][team] = "no_detections"
                match_result["status"] = "partial_failure"
                continue
            
            print(f"  [{team}] Raw detections: {len(df_raw)}")
            
            # Step B: Clean
            print(f"  [{team}] Cleaning...")
            df_clean = clean_detections(df_raw, match_id=match_id, api_key=API_KEY, routing_region=ROUTING_REGION)
            
            if df_clean is None or df_clean.empty:
                print(f"  [{team}] ✗ Cleaning failed (insufficient data).")
                match_result["teams"][team] = "cleaning_failed"
                match_result["status"] = "partial_failure"
                continue
            
            # Step C: Save
            cleaned_csv.parent.mkdir(parents=True, exist_ok=True)
            df_clean.to_csv(cleaned_csv, index=False)
            
            print(f"  [{team}] ✓ Saved {len(df_clean)} cleaned rows → {cleaned_csv.name}")
            match_result["teams"][team] = f"{len(df_clean)} rows"
    
    except Exception as e:
        # ── Mark the folder as failed by renaming it ──────────────────
        print(f"  ✗ CRASHED: {type(e).__name__}: {e}")
        traceback.print_exc()
        
        match_result["status"] = "crashed"
        match_result["error"]  = str(e)
        
        # Rename folder: SG2-12345 → _FAILED_SG2-12345
        match_dir = DATASET_ROOT / match_id
        failed_name = f"_FAILED_{match_id}"
        failed_dir  = DATASET_ROOT / failed_name
        
        if match_dir.exists() and not failed_dir.exists():
            match_dir.rename(failed_dir)
            print(f"  → Renamed folder to: {failed_name}")
        
        print(f"  → Continuing to next match...\n")
    
    results_summary.append(match_result)

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE")
print(f"{'='*60}")


[1/2] Processing: SG2-171573370
  [Blue] Running YOLO inference...
    ⚠ Screenshot folder not found: C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-171573370\0\Blue
  [Blue] ✗ No detections. Skipping.
  [All] Running YOLO inference...
    ⚠ No screenshots found in C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-171573370\0\All
  [All] ✗ No detections. Skipping.

[2/2] Processing: SG2-178899790
  [Blue] Running YOLO inference...


    YOLO (Blue):   0%|          | 0/899 [00:00<?, ?it/s]

  [Blue] Raw detections: 6446
  [Blue] Cleaning...
    ⚠️ Riot API Key is expired (401 Unauthorized).
  [Blue] ✓ Saved 9000 cleaned rows → cleaned_match_data.csv
  [All] Running YOLO inference...


  [All] Raw detections: 8786
  [All] Cleaning...
    ⚠️ Riot API Key is expired (401 Unauthorized).
  [All] ✓ Saved 9000 cleaned rows → cleaned_match_data.csv

PIPELINE COMPLETE


## 6. Summary Report

In [8]:
# ── Print summary ────────────────────────────────────────────────────
if results_summary:
    print(f"\nProcessed {len(results_summary)} matches:\n")
    
    successes = [r for r in results_summary if r['status'] == 'success']
    failures  = [r for r in results_summary if r['status'] != 'success']
    
    print(f"  ✓ Successful: {len(successes)}")
    print(f"  ✗ Issues:     {len(failures)}")
    
    if failures:
        print(f"\nMatches with issues:")
        for r in failures:
            print(f"  {r['match_id']}: {r['teams']}")
else:
    print("No matches were processed (all already cleaned, or none found).")

# Overall dataset stats
total_cleaned = 0
for match_dir in all_match_dirs:
    for team in TEAMS:
        cleaned_csv = match_dir / f"{team} data" / "cleaned_match_data.csv"
        if cleaned_csv.exists():
            total_cleaned += 1

print(f"\nTotal cleaned CSVs in Dataset: {total_cleaned} / {len(all_match_dirs) * len(TEAMS)} expected")


Processed 25 matches:

  ✓ Successful: 23
  ✗ Issues:     2

Matches with issues:
  SG2-171573370: {'Blue': 'no_detections', 'All': 'no_detections'}
  SG2-171986641: {}

Total cleaned CSVs in Dataset: 96 / 100 expected


## 7. Quick Sanity Check (Optional)

Spot-check a random cleaned match to verify the data looks correct.

In [5]:
# Pick a random cleaned match to inspect
import random

# Find matches with cleaned Blue data
cleaned_matches = [
    d.name for d in all_match_dirs 
    if (d / "Blue data" / "cleaned_match_data.csv").exists()
]

if cleaned_matches:
    sample_match = random.choice(cleaned_matches)
    sample_csv = DATASET_ROOT / sample_match / "Blue data" / "cleaned_match_data.csv"
    df_sample = pd.read_csv(sample_csv)
    
    print(f"Sanity check: {sample_match} (Blue)")
    print(f"  Rows:    {len(df_sample)}")
    print(f"  Columns: {list(df_sample.columns)}")
    print(f"  Frames:  {df_sample['frame'].min()} → {df_sample['frame'].max()}")
    print(f"  Roles:   {df_sample['role'].unique().tolist()}")
    print(f"\nFirst 10 rows:")
    print(df_sample.head(10).to_markdown())
else:
    print("No cleaned data found yet.")

Sanity check: SG2-172047870 (Blue)
  Rows:    14140
  Columns: ['frame', 'champion_id', 'champion_name', 'x', 'y', 'confidence', 'role']
  Frames:  81 → 1494
  Roles:   ['Blue_Top', 'Blue_Jungle', 'Blue_Mid', 'Blue_Bot', 'Blue_Support', 'Red_Top', 'Red_Jungle', 'Red_Mid', 'Red_Bot', 'Red_Support']

First 10 rows:
|    |   frame |   champion_id | champion_name   |     x |     y |   confidence | role         |
|---:|--------:|--------------:|:----------------|------:|------:|-------------:|:-------------|
|  0 |      81 |            40 | Gragas          |  63.6 |  87.4 |        0.918 | Blue_Top     |
|  1 |      81 |            43 | Hecarim         | 121.6 | 234   |        0.9   | Blue_Jungle  |
|  2 |      81 |            45 | Hwei            | 262.7 | 256.2 |        0.915 | Blue_Mid     |
|  3 |      81 |            76 | Lux             | 428.3 | 411.8 |        0.897 | Blue_Bot     |
|  4 |      81 |            32 | Ezreal          | 433.7 | 418.2 |        0.725 | Blue_Support |
|  5 |